# Prompt Tuning vs Prefix Tuning with SmolLM

Both methods keep the base language model completely frozen and train only a small set of new parameters:

- **Prompt Tuning**: prepends a handful of trainable "virtual token" embeddings to the input embedding sequence. Nothing else changes — the rest of the forward pass is identical to the base model.
- **Prefix Tuning**: a deeper version of the same idea. Instead of only touching the input embeddings, it injects a trainable prefix into the key/value cache of *every* attention layer (via a small reparameterizing MLP), giving the model more capacity to adapt at the cost of more trainable parameters.

We demonstrate both on a tiny sentiment-classification task, framed as text generation: given a movie review, the model should generate the single word `positive` or `negative`. We compare against the untouched base model (zero-shot) to see how much these few trainable parameters actually buy us.

In [ ]:
import random

import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)
from peft import (
    PrefixTuningConfig,
    PromptTuningConfig,
    PromptTuningInit,
    TaskType,
    get_peft_model,
)

random.seed(42)

In [ ]:
##### Step 1: Choose the base model
# SmolLM-135M-Instruct is small enough to fine-tune in a couple of minutes on a
# laptop CPU/MPS, which makes it a good fit for a "watch it actually learn" PEFT demo.
# Swap in "HuggingFaceTB/SmolLM3-3B" for a much more capable (but much slower) base model —
# everything below works unchanged, just point `checkpoint` at it.
checkpoint = "HuggingFaceTB/SmolLM-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
##### Step 2: A tiny sentiment classification task
LABELS = ["negative", "positive"]

raw_dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes")
train_dataset = raw_dataset["train"].shuffle(seed=42).select(range(200))
test_dataset = raw_dataset["test"].shuffle(seed=42).select(range(50))

train_dataset[0]

In [ ]:
##### Step 3: Build prompts, and mask the prompt tokens out of the training loss
# (we only want the model to be scored on predicting the label word, not on
# reproducing the review text it was just given)

def make_prompt(text):
    return f"Review: {text}\nSentiment:"

def preprocess(examples):
    input_ids_list, labels_list, attention_masks = [], [], []
    for text, label in zip(examples["text"], examples["label"]):
        prompt_ids = tokenizer(make_prompt(text), add_special_tokens=False)["input_ids"]
        target_ids = tokenizer(" " + LABELS[label], add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]

        input_ids_list.append(prompt_ids + target_ids)
        labels_list.append([-100] * len(prompt_ids) + target_ids)
        attention_masks.append([1] * (len(prompt_ids) + len(target_ids)))

    return {"input_ids": input_ids_list, "labels": labels_list, "attention_mask": attention_masks}

train_tokenized = train_dataset.map(preprocess, batched=True, remove_columns=train_dataset.column_names)
data_collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)

In [ ]:
##### Step 4: A simple generation-based accuracy metric

@torch.no_grad()
def evaluate_accuracy(model, dataset):
    model.eval()
    correct = 0
    for example in dataset:
        inputs = tokenizer(make_prompt(example["text"]), return_tensors="pt").to(model.device)
        output = model.generate(**inputs, max_new_tokens=3, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()

        if "positive" in generated:
            prediction = 1
        elif "negative" in generated:
            prediction = 0
        else:
            prediction = -1  # model didn't generate either label word

        correct += int(prediction == example["label"])
    return correct / len(dataset)

In [ ]:
##### Step 5: Baseline — how well does the untouched base model do, zero-shot?
base_model = AutoModelForCausalLM.from_pretrained(checkpoint, dtype=torch.float32)
baseline_accuracy = evaluate_accuracy(base_model, test_dataset)
print(f"Baseline (no tuning) accuracy: {baseline_accuracy:.2%}")

In [ ]:
##### Step 6: Prompt Tuning
prompt_tuning_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=8,
    prompt_tuning_init=PromptTuningInit.TEXT,
    prompt_tuning_init_text="Classify the sentiment of this movie review as positive or negative.",
    tokenizer_name_or_path=checkpoint,
)

prompt_model = get_peft_model(
    AutoModelForCausalLM.from_pretrained(checkpoint, dtype=torch.float32),
    prompt_tuning_config,
)
prompt_model.print_trainable_parameters()

In [ ]:
prompt_training_args = TrainingArguments(
    output_dir="../../data/models/smollm-prompt-tuned-sentiment",
    num_train_epochs=10,
    learning_rate=1e-3,
    per_device_train_batch_size=8,
    logging_strategy="epoch",
    report_to="none",
    dataloader_pin_memory=False,
    seed=42,
)

prompt_trainer = Trainer(
    model=prompt_model,
    args=prompt_training_args,
    train_dataset=train_tokenized,
    data_collator=data_collator,
)
prompt_result = prompt_trainer.train()
prompt_result

In [ ]:
prompt_tuned_accuracy = evaluate_accuracy(prompt_model, test_dataset)
print(f"Prompt-tuned accuracy: {prompt_tuned_accuracy:.2%}")

In [ ]:
##### Step 7: Prefix Tuning
prefix_tuning_config = PrefixTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=10,
    prefix_projection=True,
    encoder_hidden_size=128,
)

prefix_model = get_peft_model(
    AutoModelForCausalLM.from_pretrained(checkpoint, dtype=torch.float32),
    prefix_tuning_config,
)
prefix_model.print_trainable_parameters()

In [ ]:
prefix_training_args = TrainingArguments(
    output_dir="../../data/models/smollm-prefix-tuned-sentiment",
    num_train_epochs=10,
    learning_rate=1e-3,
    per_device_train_batch_size=8,
    logging_strategy="epoch",
    report_to="none",
    dataloader_pin_memory=False,
    seed=42,
)

prefix_trainer = Trainer(
    model=prefix_model,
    args=prefix_training_args,
    train_dataset=train_tokenized,
    data_collator=data_collator,
)
prefix_result = prefix_trainer.train()
prefix_result

In [ ]:
prefix_tuned_accuracy = evaluate_accuracy(prefix_model, test_dataset)
print(f"Prefix-tuned accuracy: {prefix_tuned_accuracy:.2%}")

In [ ]:
##### Step 8: Side-by-side comparison
prompt_trainable, prompt_total = prompt_model.get_nb_trainable_parameters()
prefix_trainable, prefix_total = prefix_model.get_nb_trainable_parameters()

comparison = pd.DataFrame({
    "Baseline (no tuning)": {
        "trainable params": 0,
        "trainable %": 0.0,
        "accuracy": baseline_accuracy,
    },
    "Prompt Tuning": {
        "trainable params": prompt_trainable,
        "trainable %": round(100 * prompt_trainable / prompt_total, 4),
        "accuracy": prompt_tuned_accuracy,
    },
    "Prefix Tuning": {
        "trainable params": prefix_trainable,
        "trainable %": round(100 * prefix_trainable / prefix_total, 4),
        "accuracy": prefix_tuned_accuracy,
    },
})
comparison

In [ ]:
##### Step 9: A few qualitative examples
for example in test_dataset.select(range(5)):
    prompt = make_prompt(example["text"])
    inputs = tokenizer(prompt, return_tensors="pt")
    print(example["text"][:80], "...")
    print("  true:", LABELS[example["label"]])
    for name, m in [("prompt-tuned", prompt_model), ("prefix-tuned", prefix_model)]:
        m_inputs = inputs.to(m.device)
        out = m.generate(**m_inputs, max_new_tokens=3, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        gen = tokenizer.decode(out[0][m_inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        print(f"  {name} generated: {gen!r}")
    print()